# Validation

In [ ]:
# 03_validate_rag_lora_or_qlora.py
# RAG validation/inference stage.
#
# This script:
#   1. Loads validation tickets.
#   2. Retrieves from train-only FAISS using BAAI/bge-m3.
#   3. Reranks with BAAI/bge-reranker-v2-m3 using a safe manual wrapper.
#   4. Loads the fine-tuned Llama LoRA/QLoRA adapter.
#   5. Generates answers and saves predictions/metrics.
#
# Install:
#   pip install -U pandas numpy tqdm ftfy faiss-cpu FlagEmbedding transformers peft bitsandbytes sentencepiece protobuf safetensors huggingface_hub

import gc
import json
import os
import random
import re
import getpass
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence

import faiss
import numpy as np
import pandas as pd
import torch
from huggingface_hub import get_token, login
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from FlagEmbedding import BGEM3FlagModel

try:
    import ftfy
except Exception:
    ftfy = None


@dataclass
class Config:
    output_dir: str = r"C:\Users\XXXXXXXXXX\Project_RAG\outputs"

    generator_model_id: str = "meta-llama/Llama-3.1-8B-Instruct"
    embedding_model_id: str = "BAAI/bge-m3"
    reranker_model_id: str = "BAAI/bge-reranker-v2-m3"

    # If True, ask for token every run. If False, use HF_TOKEN or cached Hugging Face login first.
    force_token_prompt: bool = False

    seed: int = 42

    # CPU generation with Llama is slow. Set None for all validation rows.
    max_validation_examples: Optional[int] = 100

    faiss_candidate_k: int = 30
    rerank_top_k: int = 5

    embedding_batch_size: int = 16
    embedding_max_length: int = 8192

    reranker_batch_size: int = 8
    reranker_max_length: int = 512
    reranker_max_passage_chars: int = 3500

    max_context_chars_per_ticket: int = 900
    max_query_chars: int = 3500
    max_prompt_tokens: int = 4096
    max_new_tokens: int = 384

    # CPU dtype. "float16" uses less RAM. If it errors, try "float32".
    cpu_torch_dtype: str = "float16"
    cpu_allow_float32_fallback: bool = True


CFG = Config()

SYSTEM_PROMPT = (
    "You are a multilingual IT customer-support assistant. "
    "Use the retrieved solved tickets as grounding evidence. "
    "Answer in the same language as the customer ticket unless the customer asks otherwise. "
    "Be concise, professional, and action-oriented. "
    "If the retrieved evidence is insufficient, ask for the missing details instead of inventing facts, policies, ETAs, prices, or account-specific information."
)

MOJIBAKE_MARKERS = ("Ã", "Â", "â€", "â€™", "â€œ", "â€\x9d", "ðŸ")


class SafeBGEReranker:
    """
    Manual reranker wrapper for BAAI/bge-reranker-v2-m3.
    This avoids the fragile FlagEmbedding tokenizer.prepare_for_model path.
    """

    def __init__(
        self,
        model_id: str,
        batch_size: int = 8,
        max_length: int = 512,
        use_fp16: bool = True,
    ) -> None:
        self.model_id = model_id
        self.batch_size = batch_size
        self.max_length = max_length
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        token = os.environ.get("HF_TOKEN") or get_token()

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=token,
            use_fast=True,
        )

        dtype = torch.float16 if torch.cuda.is_available() and use_fp16 else torch.float32

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_id,
            token=token,
            torch_dtype=dtype,
        )

        self.model.to(self.device)
        self.model.eval()

    @torch.inference_mode()
    def compute_score(self, pairs: List[List[str]], normalize: bool = True) -> List[float]:
        if not pairs:
            return []

        scores: List[float] = []

        for start in range(0, len(pairs), self.batch_size):
            batch = pairs[start : start + self.batch_size]
            queries = [p[0] for p in batch]
            passages = [p[1] for p in batch]

            inputs = self.tokenizer(
                queries,
                passages,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )

            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            logits = self.model(**inputs).logits

            if logits.ndim == 2:
                if logits.shape[1] == 1:
                    logits = logits[:, 0]
                else:
                    logits = logits[:, -1]
            else:
                logits = logits.view(-1)

            if normalize:
                logits = torch.sigmoid(logits)

            scores.extend(logits.detach().float().cpu().tolist())

        return scores


def ask_for_hf_token(cfg: Config) -> str:
    cached = os.environ.get("HF_TOKEN") or get_token()

    if cfg.force_token_prompt or not cached:
        token = getpass.getpass("Paste your Hugging Face token. It will not be shown: ").strip()

        if not token:
            raise RuntimeError("No Hugging Face token was entered.")

        os.environ["HF_TOKEN"] = token
        login(token=token, add_to_git_credential=False)

        return token

    os.environ["HF_TOKEN"] = cached
    return cached


def dtype_from_name(name: str) -> torch.dtype:
    name = name.lower().strip()

    if name in {"float16", "fp16", "half"}:
        return torch.float16

    if name in {"bfloat16", "bf16"}:
        return torch.bfloat16

    if name in {"float32", "fp32"}:
        return torch.float32

    raise ValueError(f"Unsupported dtype name: {name}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def paths(cfg: Config) -> Dict[str, Path]:
    out = Path(cfg.output_dir)

    return {
        "output": out,
        "processed": out / "processed",
        "rag_store": out / "rag_store",
        "adapter": out / "llama31_8b_rag_lora_or_qlora_adapter",
        "validation_results": out / "validation_results_lora_or_qlora",
    }


def repair_text(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value)
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\ufeff", "").strip()

    if any(marker in text for marker in MOJIBAKE_MARKERS):
        if ftfy is not None:
            text = ftfy.fix_text(text)
        else:
            try:
                repaired = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
                if sum(repaired.count(m) for m in MOJIBAKE_MARKERS) < sum(text.count(m) for m in MOJIBAKE_MARKERS):
                    text = repaired
            except Exception:
                pass

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def truncate_text(text: Any, max_chars: int) -> str:
    text = repair_text(text)

    if len(text) <= max_chars:
        return text

    return text[: max_chars - 20].rstrip() + " ...[truncated]"


def extract_tags_from_row(row: pd.Series) -> List[str]:
    tags: List[str] = []

    for i in range(1, 10):
        val = repair_text(row.get(f"tag_{i}", ""))
        if val:
            tags.append(val)

    seen = set()
    clean = []

    for tag in tags:
        key = tag.lower()

        if key not in seen:
            clean.append(tag)
            seen.add(key)

    return clean


def ticket_query_text(row: pd.Series, cfg: Config) -> str:
    tags = ", ".join(extract_tags_from_row(row))

    parts = [
        f"Subject: {repair_text(row.get('subject', ''))}",
        f"Customer issue: {repair_text(row.get('body', ''))}",
        f"Type: {repair_text(row.get('type', ''))}",
        f"Queue: {repair_text(row.get('queue', ''))}",
        f"Priority: {repair_text(row.get('priority', ''))}",
        f"Language: {repair_text(row.get('language', ''))}",
        f"Business type: {repair_text(row.get('business_type', ''))}",
        f"Tags: {tags}",
    ]

    return truncate_text("\n".join([p for p in parts if not p.endswith(": ")]), cfg.max_query_chars)


def format_retrieved_context(retrieved_docs: List[Dict[str, Any]], cfg: Config) -> str:
    if not retrieved_docs:
        return "No sufficiently similar solved tickets were retrieved."

    blocks: List[str] = []

    for rank, doc in enumerate(retrieved_docs, start=1):
        tags = ", ".join(doc.get("tags", []))
        score = doc.get("rerank_score", doc.get("faiss_score", 0.0))

        block = (
            f"[Retrieved ticket {rank}]\n"
            f"Ticket ID: {doc.get('ticket_id', '')}\n"
            f"Relevance score: {score:.4f}\n"
            f"Subject: {truncate_text(doc.get('subject', ''), 250)}\n"
            f"Customer issue: {truncate_text(doc.get('body', ''), cfg.max_context_chars_per_ticket)}\n"
            f"Resolved support answer: {truncate_text(doc.get('answer', ''), cfg.max_context_chars_per_ticket)}\n"
            f"Metadata: type={doc.get('type', '')}; queue={doc.get('queue', '')}; priority={doc.get('priority', '')}; "
            f"language={doc.get('language', '')}; business_type={doc.get('business_type', '')}; tags={tags}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)


def build_user_prompt(row: pd.Series, retrieved_docs: List[Dict[str, Any]], cfg: Config) -> str:
    tags = ", ".join(extract_tags_from_row(row))
    context = format_retrieved_context(retrieved_docs, cfg)

    return (
        "Customer ticket to answer:\n"
        f"Subject: {truncate_text(row.get('subject', ''), 300)}\n"
        f"Message:\n{truncate_text(row.get('body', ''), cfg.max_query_chars)}\n\n"
        "Ticket metadata:\n"
        f"type={repair_text(row.get('type', ''))}; queue={repair_text(row.get('queue', ''))}; "
        f"priority={repair_text(row.get('priority', ''))}; language={repair_text(row.get('language', ''))}; "
        f"business_type={repair_text(row.get('business_type', ''))}; tags={tags}\n\n"
        "Retrieved solved-ticket evidence:\n"
        f"{context}\n\n"
        "Task: Draft the best customer-support response grounded in the retrieved evidence. "
        "Do not mention internal model behavior. Do not expose private placeholders except when asking the customer to provide missing account/device/log details."
    )


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def encode_dense(
    model: BGEM3FlagModel,
    texts: List[str],
    cfg: Config,
    desc: str,
) -> np.ndarray:
    vectors: List[np.ndarray] = []

    for start in tqdm(range(0, len(texts), cfg.embedding_batch_size), desc=desc):
        batch = texts[start : start + cfg.embedding_batch_size]
        output = model.encode(batch, batch_size=cfg.embedding_batch_size, max_length=cfg.embedding_max_length)
        dense = np.asarray(output["dense_vecs"], dtype="float32")
        vectors.append(dense)

    arr = np.vstack(vectors).astype("float32")
    faiss.normalize_L2(arr)

    return arr


def faiss_candidates_for_row(
    ids: Sequence[int],
    scores: Sequence[float],
    corpus: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []

    for idx, score in zip(ids, scores):
        if int(idx) < 0:
            continue

        doc = dict(corpus[int(idx)])
        doc["faiss_score"] = float(score)
        candidates.append(doc)

    return candidates


def rerank_candidates(
    reranker: SafeBGEReranker,
    query_text: str,
    candidates: List[Dict[str, Any]],
    cfg: Config,
) -> List[Dict[str, Any]]:
    if not candidates:
        return []

    pairs = [
        [query_text, truncate_text(doc.get("context_text", ""), cfg.reranker_max_passage_chars)]
        for doc in candidates
    ]

    scores = reranker.compute_score(pairs, normalize=True)

    reranked: List[Dict[str, Any]] = []

    for doc, score in zip(candidates, scores):
        updated = dict(doc)
        updated["rerank_score"] = float(score)
        reranked.append(updated)

    reranked.sort(key=lambda d: d["rerank_score"], reverse=True)

    return reranked[: cfg.rerank_top_k]


def load_validation_tokenizer(cfg: Config, p: Dict[str, Path], token: str) -> AutoTokenizer:
    tokenizer_source = str(p["adapter"]) if (p["adapter"] / "tokenizer_config.json").exists() else cfg.generator_model_id

    tokenizer = AutoTokenizer.from_pretrained(
        tokenizer_source,
        token=token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    return tokenizer


def load_validation_model(cfg: Config, p: Dict[str, Path], token: str) -> PeftModel:
    if torch.cuda.is_available():
        bf16 = torch.cuda.is_bf16_supported()
        compute_dtype = torch.bfloat16 if bf16 else torch.float16

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_use_double_quant=True,
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            cfg.generator_model_id,
            token=token,
            quantization_config=quant_config,
            torch_dtype=compute_dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
    else:
        cpu_dtype = dtype_from_name(cfg.cpu_torch_dtype)

        try:
            base_model = AutoModelForCausalLM.from_pretrained(
                cfg.generator_model_id,
                token=token,
                torch_dtype=cpu_dtype,
                low_cpu_mem_usage=True,
            )
        except Exception as exc:
            if cfg.cpu_allow_float32_fallback and cpu_dtype != torch.float32:
                print(f"CPU model load with {cfg.cpu_torch_dtype} failed. Retrying with float32.")
                print(f"Original error: {exc}")

                base_model = AutoModelForCausalLM.from_pretrained(
                    cfg.generator_model_id,
                    token=token,
                    torch_dtype=torch.float32,
                    low_cpu_mem_usage=True,
                )
            else:
                raise

    model = PeftModel.from_pretrained(base_model, str(p["adapter"]))
    model.eval()

    return model


def model_input_device(model: PeftModel) -> torch.device:
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def eos_token_ids(tokenizer: AutoTokenizer) -> List[int]:
    ids: List[int] = []

    if tokenizer.eos_token_id is not None:
        ids.append(int(tokenizer.eos_token_id))

    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

    if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in ids:
        ids.append(eot_id)

    return ids


def generate_support_answer(
    row: pd.Series,
    retrieved_docs: List[Dict[str, Any]],
    model: PeftModel,
    tokenizer: AutoTokenizer,
    cfg: Config,
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row, retrieved_docs, cfg)},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.max_prompt_tokens,
    )

    device = model_input_device(model)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=cfg.max_new_tokens,
            do_sample=False,
            eos_token_id=eos_token_ids(tokenizer),
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1] :]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    return repair_text(text)


def word_tokens(text: str) -> List[str]:
    return re.findall(r"\w+", repair_text(text).lower(), flags=re.UNICODE)


def lexical_f1(prediction: str, reference: str) -> float:
    pred = word_tokens(prediction)
    ref = word_tokens(reference)

    if not pred and not ref:
        return 1.0

    if not pred or not ref:
        return 0.0

    pred_counts = Counter(pred)
    ref_counts = Counter(ref)
    overlap = sum((pred_counts & ref_counts).values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred)
    recall = overlap / len(ref)

    return 2 * precision * recall / (precision + recall)


def lcs_length(a: List[str], b: List[str]) -> int:
    if not a or not b:
        return 0

    prev = [0] * (len(b) + 1)

    for token_a in a:
        curr = [0]

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                curr.append(prev[j - 1] + 1)
            else:
                curr.append(max(prev[j], curr[-1]))

        prev = curr

    return prev[-1]


def rouge_l_f1(prediction: str, reference: str) -> float:
    pred = word_tokens(prediction)
    ref = word_tokens(reference)

    if not pred and not ref:
        return 1.0

    if not pred or not ref:
        return 0.0

    lcs = lcs_length(pred, ref)

    if lcs == 0:
        return 0.0

    precision = lcs / len(pred)
    recall = lcs / len(ref)

    return 2 * precision * recall / (precision + recall)


def normalized_set(values: Iterable[str]) -> set:
    return {repair_text(v).lower() for v in values if repair_text(v)}


def retrieval_metadata_metrics(row: pd.Series, docs: List[Dict[str, Any]]) -> Dict[str, float]:
    gold_queue = repair_text(row.get("queue", "")).lower()
    gold_language = repair_text(row.get("language", "")).lower()
    gold_tags = normalized_set(extract_tags_from_row(row))

    top1 = docs[:1]
    topk = docs

    retrieved_tags = set()

    for doc in topk:
        retrieved_tags.update(normalized_set(doc.get("tags", [])))

    return {
        "top1_same_queue": float(bool(top1 and repair_text(top1[0].get("queue", "")).lower() == gold_queue)),
        "top1_same_language": float(bool(top1 and repair_text(top1[0].get("language", "")).lower() == gold_language)),
        "topk_same_queue": float(any(repair_text(doc.get("queue", "")).lower() == gold_queue for doc in topk)),
        "topk_same_language": float(any(repair_text(doc.get("language", "")).lower() == gold_language for doc in topk)),
        "topk_tag_overlap": float(bool(gold_tags and retrieved_tags and (gold_tags & retrieved_tags))),
        "best_rerank_score": float(topk[0].get("rerank_score", 0.0)) if topk else 0.0,
        "best_faiss_score": float(topk[0].get("faiss_score", 0.0)) if topk else 0.0,
    }


def summarize_metrics(metric_rows: List[Dict[str, float]]) -> Dict[str, float]:
    if not metric_rows:
        return {"n": 0.0}

    keys = sorted(metric_rows[0].keys())
    summary: Dict[str, float] = {"n": float(len(metric_rows))}

    for key in keys:
        values = [row[key] for row in metric_rows if key in row]
        summary[f"mean_{key}"] = float(np.mean(values)) if values else 0.0

    return summary


def validate(cfg: Config) -> None:
    set_seed(cfg.seed)

    p = paths(cfg)
    p["validation_results"].mkdir(parents=True, exist_ok=True)

    token = ask_for_hf_token(cfg)

    val_csv = p["processed"] / "validation.csv"
    index_path = p["rag_store"] / "faiss_train.index"
    corpus_path = p["rag_store"] / "corpus.jsonl"
    adapter_config = p["adapter"] / "adapter_config.json"

    for required in (val_csv, index_path, corpus_path, adapter_config):
        if not required.exists():
            raise FileNotFoundError(f"Missing required artifact: {required}")

    with open(p["validation_results"] / "config_validate_lora_or_qlora.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)

    print("Loading validation data, FAISS index, and train corpus...")
    val_df = pd.read_csv(val_csv).fillna("")

    if cfg.max_validation_examples is not None and len(val_df) > cfg.max_validation_examples:
        val_df = val_df.sample(n=cfg.max_validation_examples, random_state=cfg.seed).reset_index(drop=True)

    corpus = read_jsonl(corpus_path)
    index = faiss.read_index(str(index_path))

    print("Loading BGE-M3 embedder and encoding validation queries...")
    embedder = BGEM3FlagModel(cfg.embedding_model_id, use_fp16=torch.cuda.is_available())

    query_texts = [ticket_query_text(row, cfg) for _, row in val_df.iterrows()]

    query_vectors = encode_dense(
        embedder,
        query_texts,
        cfg=cfg,
        desc="Embedding validation queries",
    )

    faiss_scores, faiss_ids = index.search(query_vectors, cfg.faiss_candidate_k)

    del embedder, query_vectors
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Loading BGE reranker...")
    reranker = SafeBGEReranker(
        cfg.reranker_model_id,
        batch_size=cfg.reranker_batch_size,
        max_length=cfg.reranker_max_length,
        use_fp16=torch.cuda.is_available(),
    )

    all_retrieved: List[List[Dict[str, Any]]] = []

    for i, row in tqdm(list(val_df.iterrows()), desc="Retrieve + rerank"):
        candidates = faiss_candidates_for_row(faiss_ids[i], faiss_scores[i], corpus)

        retrieved = rerank_candidates(
            reranker=reranker,
            query_text=query_texts[i],
            candidates=candidates,
            cfg=cfg,
        )

        all_retrieved.append(retrieved)

    del reranker
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Loading fine-tuned Llama adapter...")
    tokenizer = load_validation_tokenizer(cfg, p, token)
    model = load_validation_model(cfg, p, token)

    metric_rows: List[Dict[str, float]] = []
    csv_rows: List[Dict[str, Any]] = []
    jsonl_rows: List[Dict[str, Any]] = []

    print("Generating validation answers...")

    for i, row in tqdm(list(val_df.iterrows()), desc="Generate"):
        retrieved = all_retrieved[i]

        generated_answer = generate_support_answer(
            row=row,
            retrieved_docs=retrieved,
            model=model,
            tokenizer=tokenizer,
            cfg=cfg,
        )

        gold_answer = repair_text(row.get("answer", ""))

        metrics = {
            "lexical_f1": lexical_f1(generated_answer, gold_answer),
            "rouge_l_f1": rouge_l_f1(generated_answer, gold_answer),
            **retrieval_metadata_metrics(row, retrieved),
        }

        metric_rows.append(metrics)

        retrieved_brief = [
            {
                "rank": rank,
                "ticket_id": doc.get("ticket_id", ""),
                "subject": doc.get("subject", ""),
                "queue": doc.get("queue", ""),
                "language": doc.get("language", ""),
                "tags": doc.get("tags", []),
                "faiss_score": doc.get("faiss_score", 0.0),
                "rerank_score": doc.get("rerank_score", 0.0),
            }
            for rank, doc in enumerate(retrieved, start=1)
        ]

        jsonl_row = {
            "ticket_id": repair_text(row.get("ticket_id", "")),
            "subject": repair_text(row.get("subject", "")),
            "language": repair_text(row.get("language", "")),
            "queue": repair_text(row.get("queue", "")),
            "priority": repair_text(row.get("priority", "")),
            "customer_body": repair_text(row.get("body", "")),
            "gold_answer": gold_answer,
            "generated_answer": generated_answer,
            "metrics": metrics,
            "retrieved": retrieved_brief,
        }

        jsonl_rows.append(jsonl_row)

        csv_row = {
            "ticket_id": jsonl_row["ticket_id"],
            "subject": jsonl_row["subject"],
            "language": jsonl_row["language"],
            "queue": jsonl_row["queue"],
            "priority": jsonl_row["priority"],
            "gold_answer": gold_answer,
            "generated_answer": generated_answer,
            "retrieved_ticket_ids": ",".join([doc["ticket_id"] for doc in retrieved_brief]),
        }

        csv_row.update(metrics)
        csv_rows.append(csv_row)

    predictions_jsonl = p["validation_results"] / "predictions.jsonl"
    predictions_csv = p["validation_results"] / "predictions.csv"
    metrics_json = p["validation_results"] / "metrics_summary.json"

    write_jsonl(predictions_jsonl, jsonl_rows)

    pd.DataFrame(csv_rows).to_csv(
        predictions_csv,
        index=False,
        encoding="utf-8",
    )

    summary = summarize_metrics(metric_rows)

    with open(metrics_json, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nValidation complete.")
    print(f"Predictions JSONL: {predictions_jsonl}")
    print(f"Predictions CSV:   {predictions_csv}")
    print(f"Metrics summary:   {metrics_json}")

    print("\nSummary metrics:")

    for key, value in summary.items():
        if key == "n":
            print(f"  {key}: {int(value)}")
        else:
            print(f"  {key}: {value:.4f}")


if __name__ == "__main__":
    validate(CFG)